In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, AllChem, rdMolTransforms
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DIM_F1 = 20
DIM_F2_RAW = 1024
DIM_F2_PCA = 128
DIM_F3 = 10
DIM_F4 = 43
DIM_HB = 2
DIM_ONE_ASSAY = 3
DIM_INTER = 14

MODAL_DIMS_SINGLE = [DIM_F1, DIM_F2_PCA, DIM_F3, DIM_F4, DIM_HB, DIM_ONE_ASSAY]
FUSION_HIDDEN = 192
HA_HB_DIM = 192
GAN_INPUT_DIM = HA_HB_DIM * 2 + DIM_INTER

EMBED_DIM = 32
EXP_DIM = 32
INPUT_DIM = GAN_INPUT_DIM + EMBED_DIM + EXP_DIM

NUM_CLASSES = 5
label_map = {
    "Antagonistic effect": 0,
    "Synergistic effect": 1,
    "Not mentioned": 2,
    "Additive effect": 3,
    "Independent effect": 4
}
inv_label_map = {v: k for k, v in label_map.items()}
class_names = list(inv_label_map.values())

ENDO_ASSAY_COLS = [
    "er_pred_hitc-SMILES1", "ar_pred_hitc-SMILES1", "cyp19_pred_hitc-SMILES1",
    "er_pred_hitc-SMILES2", "ar_pred_hitc-SMILES2", "cyp19_pred_hitc-SMILES2"
]
NEURO_ASSAY_COLS = [
    "neuro1_pred_hitc-SMILES1", "neuro2_pred_hitc-SMILES1", "neuro3_pred_hitc-SMILES1",
    "neuro1_pred_hitc-SMILES2", "neuro2_pred_hitc-SMILES2", "neuro3_pred_hitc-SMILES2"
]
HEP_ASSAY_COLS = [
    "hep1_pred_hitc-SMILES1", "hep2_pred_hitc-SMILES1", "hep3_pred_hitc-SMILES1",
    "hep1_pred_hitc-SMILES2", "hep2_pred_hitc-SMILES2", "hep3_pred_hitc-SMILES2"
]
IMM_ASSAY_COLS = [
    "imm1_pred_positive_prob-SMILES1", "imm2_pred_positive_prob-SMILES1", "imm3_pred_positive_prob-SMILES1",
    "imm1_pred_positive_prob-SMILES2", "imm2_pred_positive_prob-SMILES2", "imm3_pred_positive_prob-SMILES2"
]
DEV_ASSAY_COLS = [
    "dev1_pred_hitc-SMILES1", "dev2_pred_hitc-SMILES1", "dev3_pred_hitc-SMILES1",
    "dev1_pred_hitc-SMILES2", "dev2_pred_hitc-SMILES2", "dev3_pred_hitc-SMILES2"
]

MODEL_CONFIGS = [
    {
        "name": "endo",
        "ckpt_path": "reproduction_model.pth",
        "assay_cols": ENDO_ASSAY_COLS
    },
    {
        "name": "neuro",
        "ckpt_path": "neurotoxicity_model.pth",
        "assay_cols": NEURO_ASSAY_COLS
    },
    {
        "name": "hep",
        "ckpt_path": "hepatotoxicity_model.pth",
        "assay_cols": HEP_ASSAY_COLS
    },
    {
        "name": "imm",
        "ckpt_path": "immunotoxicity_model.pth",
        "assay_cols": IMM_ASSAY_COLS
    },
    {
        "name": "dev",
        "ckpt_path": "developmental-toxicity_model.pth",
        "assay_cols": DEV_ASSAY_COLS
    }
]

def to_fixed_len(arr, target_len):
    arr = np.asarray(arr, dtype=np.float32).ravel()
    if len(arr) < target_len:
        arr = np.concatenate([arr, np.zeros(target_len - len(arr), dtype=np.float32)])
    else:
        arr = arr[:target_len]
    return arr

def generate_3d_conformer(mol, max_attempts=10):
    try:
        mol_3d = Chem.AddHs(mol)
        for _ in range(max_attempts):
            if AllChem.EmbedMolecule(mol_3d, randomSeed=42) == 0:
                AllChem.MMFFOptimizeMolecule(mol_3d)
                return mol_3d
        return None
    except:
        return None

def get_3d_geometric_features(mol_3d):
    feats = []
    if mol_3d is not None and mol_3d.GetNumConformers() > 0:
        try:
            conf = mol_3d.GetConformer()
            num_atoms = mol_3d.GetNumAtoms()
            coords = np.array([[conf.GetAtomPosition(i).x, conf.GetAtomPosition(i).y, conf.GetAtomPosition(i).z]
                               for i in range(num_atoms)])
            feats.extend(coords.mean(axis=0).tolist())
            feats.extend(coords.std(axis=0).tolist())
            feats.extend(coords.max(axis=0).tolist())
            feats.extend(coords.min(axis=0).tolist())

            bond_lengths = [rdMolTransforms.GetBondLength(conf, b.GetBeginAtomIdx(), b.GetEndAtomIdx())
                            for b in mol_3d.GetBonds()]
            feats += [np.mean(bond_lengths), np.std(bond_lengths), np.max(bond_lengths), np.min(bond_lengths)] if bond_lengths else [0.0]*4

            angles = []
            for a in range(num_atoms):
                for n in mol_3d.GetAtomWithIdx(a).GetNeighbors():
                    angles.append(rdMolTransforms.GetAngleDeg(conf, a, n.GetIdx(), a))
            feats += [np.mean(angles), np.std(angles), np.max(angles), np.min(angles)] if angles else [0.0]*4

            dihedrals = []
            for b in mol_3d.GetBonds():
                a1, a2 = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
                for n1 in mol_3d.GetAtomWithIdx(a1).GetNeighbors():
                    for n2 in mol_3d.GetAtomWithIdx(a2).GetNeighbors():
                        if n1.GetIdx() != a2 and n2.GetIdx() != a1:
                            dihedrals.append(rdMolTransforms.GetDihedralDeg(conf, n1.GetIdx(), a1, a2, n2.GetIdx()))
            feats += dihedrals[:20]
        except:
            pass
    return to_fixed_len(feats, DIM_F4)

def extract_mol_features_raw(smi):
    f1 = np.zeros(DIM_F1, dtype=np.float32)
    f2_raw = np.zeros(DIM_F2_RAW, dtype=np.float32)
    f3 = np.zeros(DIM_F3, dtype=np.float32)
    f4 = np.zeros(DIM_F4, dtype=np.float32)
    hb = np.zeros(DIM_HB, dtype=np.float32)

    if pd.isna(smi) or not isinstance(smi, str) or len(smi.strip()) == 0:
        return f1, f2_raw, f3, f4, hb

    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return f1, f2_raw, f3, f4, hb

    f1 = np.array([
        Descriptors.MolWt(mol), Descriptors.MolLogP(mol), Descriptors.TPSA(mol),
        Descriptors.NumHDonors(mol), Descriptors.NumHAcceptors(mol),
        Descriptors.NumValenceElectrons(mol), Descriptors.HeavyAtomCount(mol),
        Descriptors.NHOHCount(mol), Descriptors.NOCount(mol), Descriptors.FractionCSP3(mol),
        Descriptors.NumAromaticRings(mol), Descriptors.NumAliphaticRings(mol),
        Descriptors.NumSaturatedRings(mol), Descriptors.NumHeteroatoms(mol),
        Descriptors.NumRotatableBonds(mol),
        Descriptors.MaxPartialCharge(mol) if Descriptors.MaxPartialCharge(mol) else 0.0,
        Descriptors.MinPartialCharge(mol) if Descriptors.MinPartialCharge(mol) else 0.0,
        Descriptors.MolMR(mol), 0.0, 0.0,
    ], dtype=np.float32)
    f1 = to_fixed_len(f1, DIM_F1)

    try:
        fp_bit = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=DIM_F2_RAW)
        f2_raw = np.array(fp_bit, dtype=np.float32)
    except:
        f2_raw = np.zeros(DIM_F2_RAW, dtype=np.float32)
    f2_raw = to_fixed_len(f2_raw, DIM_F2_RAW)

    f3 = np.array([
        Descriptors.MolWt(mol) * 0.1, Descriptors.MolLogP(mol) * 0.5, Descriptors.TPSA(mol) * 0.01,
        Descriptors.NumHDonors(mol), Descriptors.NumHAcceptors(mol),
        Descriptors.NumAromaticRings(mol), Descriptors.NumAliphaticRings(mol),
        Descriptors.NumHeteroatoms(mol), Descriptors.NumRotatableBonds(mol),
        Descriptors.LabuteASA(mol) if Descriptors.LabuteASA(mol) else 0.0,
    ], dtype=np.float32)
    f3 = to_fixed_len(f3, DIM_F3)

    mol_3d = generate_3d_conformer(mol)
    f4 = get_3d_geometric_features(mol_3d)

    h_don = Descriptors.NumHDonors(mol)
    h_acc = Descriptors.NumHAcceptors(mol)
    hb = np.array([h_don, h_acc], dtype=np.float32)
    hb = to_fixed_len(hb, DIM_HB)

    return f1, f2_raw, f3, f4, hb

def calc_mol_mixed_interaction(f1a, f1b, f3a, f3b, f4a, f4b, hb_a, hb_b):
    def cos_sim(u, v):
        n1, n2 = np.linalg.norm(u), np.linalg.norm(v)
        return 0.0 if (n1 < 1e-8 or n2 < 1e-8) else np.dot(u, v) / (n1 * n2)
    def l2_dist(u, v):
        return np.linalg.norm(u - v)

    cos1 = cos_sim(f1a, f1b)
    dist1 = l2_dist(f1a, f1b)
    had1 = (f1a * f1b).mean()

    cos3 = cos_sim(f4a, f4b)
    dist3 = l2_dist(f4a, f4b)
    had3 = (f4a * f4b).mean()

    had_low = (f3a * f3b).mean()

    da, aa = hb_a[0], hb_a[1]
    db, ab = hb_b[0], hb_b[1]
    sum_d, sum_a = da + db, aa + ab
    diff_d, diff_a = abs(da - db), abs(aa - ab)
    mul_d, mul_a = da * db, aa * ab

    inter = np.array([
        cos1, dist1, had1, cos3, dist3, had3, had_low,
        sum_d, sum_a, diff_d, diff_a, mul_d, mul_a, 0.0
    ], dtype=np.float32)
    return to_fixed_len(inter, DIM_INTER)

class CrossInteractAttnFusion(nn.Module):
    def __init__(self, modal_dims, hidden_dim, out_dim, head=4):
        super().__init__()
        self.modal_dims = modal_dims
        self.num_modal = len(modal_dims)
        self.head = head
        self.head_dim = hidden_dim // head
        self.hidden_dim = hidden_dim
        self.out_dim = out_dim

        self.projs = nn.ModuleList([nn.Linear(d, hidden_dim) for d in modal_dims])
        self.ln_proj = nn.LayerNorm(hidden_dim)

        self.qkv_global = nn.Linear(hidden_dim, hidden_dim * 3)

        self.assay_gate = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.Sigmoid()
        )

        self.cross_q = nn.Linear(hidden_dim, hidden_dim)
        self.cross_k = nn.Linear(hidden_dim, hidden_dim)
        self.cross_v = nn.Linear(hidden_dim, hidden_dim)
        self.ln_cross = nn.LayerNorm(hidden_dim)

        self.concat_ln = nn.LayerNorm(hidden_dim * self.num_modal)
        self.out_linear = nn.Linear(hidden_dim * self.num_modal, out_dim)
        self.drop = nn.Dropout(0.35)

    def forward(self, modal_list):
        bs = modal_list[0].shape[0]
        hidden_list = []
        for idx, m in enumerate(modal_list):
            h = self.projs[idx](m)
            h = self.ln_proj(h)
            hidden_list.append(h)
        stack = torch.stack(hidden_list, dim=1)

        qkv = self.qkv_global(stack).reshape(bs, self.num_modal, 3, self.head, self.head_dim).permute(2,0,3,1,4)
        q_g, k_g, v_g = qkv[0], qkv[1], qkv[2]
        attn_score = torch.matmul(q_g, k_g.transpose(-1,-2)) / np.sqrt(self.head_dim)
        attn_weight = F.softmax(attn_score, dim=-1)
        global_attn_out = torch.matmul(attn_weight, v_g).transpose(1,2).reshape(bs, self.num_modal, self.hidden_dim)

        assay_feat = stack[:, -1, :]
        gate_w = self.assay_gate(assay_feat).unsqueeze(1)
        gated_feat = global_attn_out * gate_w

        molA_feat = gated_feat[:, :5, :].mean(dim=1, keepdim=True)
        molB_feat = gated_feat[:, :5, :].mean(dim=1, keepdim=True)
        q_c = self.cross_q(molA_feat)
        k_c = self.cross_k(molB_feat)
        v_c = self.cross_v(molB_feat)
        cross_attn = F.softmax(torch.matmul(q_c, k_c.transpose(-1,-2)) / np.sqrt(self.head_dim), dim=-1)
        cross_out = torch.matmul(cross_attn, v_c)
        cross_out = self.ln_cross(cross_out)
        fused_all = gated_feat + cross_out
        concat = fused_all.reshape(bs, -1)
        concat = self.concat_ln(concat)
        out = self.drop(self.out_linear(concat))
        return out

class ResBlock(nn.Module):
    def __init__(self, dim, dropout=0.4):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim)
        self.bn1 = nn.BatchNorm1d(dim)
        self.fc2 = nn.Linear(dim, dim)
        self.bn2 = nn.BatchNorm1d(dim)
        self.dropout = nn.Dropout(dropout)
        self.act = nn.GELU()
    def forward(self, x):
        residual = x
        out = self.dropout(self.act(self.bn1(self.fc1(x))))
        out = self.bn2(self.fc2(out))
        out += residual
        return self.act(out)

class ImprovedToxicModel(nn.Module):
    def __init__(self, total_feat_dim, embed_dim, exp_dim, species_num, num_classes):
        super().__init__()
        self.embed_dim = embed_dim
        self.exp_dim = exp_dim
        self.total_feat_dim = total_feat_dim
        self.species_emb = nn.Embedding(species_num, embed_dim)
        self.exp_proj = nn.Linear(1, exp_dim)
        self.backbone = nn.Sequential(
            nn.Linear(INPUT_DIM, 1024),
            nn.BatchNorm1d(1024),
            nn.GELU(),
            nn.Dropout(0.45),
            ResBlock(1024),
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(0.45),
            ResBlock(512),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(0.4),
            ResBlock(256),
        )
        self.classifier = nn.Linear(256, num_classes)
        self.feature_out_dim = 256
    def forward(self, full_feat, species, exposure):
        sp_feat = self.species_emb(species)
        exp_feat = F.gelu(self.exp_proj(exposure)).squeeze(1)
        all_feat = torch.cat([full_feat, sp_feat, exp_feat], dim=-1)
        hidden = self.backbone(all_feat)
        logits = self.classifier(hidden)
        return logits, hidden

def load_model_ckpt(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=device)
    pca = ckpt["pca"]
    scaler = ckpt["scaler"]
    exp_scaler = ckpt["exp_scaler"]
    sp_enc = ckpt["sp_enc"]
    species_num = len(sp_enc.classes_)
    fusion_net = CrossInteractAttnFusion(MODAL_DIMS_SINGLE, FUSION_HIDDEN, HA_HB_DIM).to(device)
    fusion_net.load_state_dict(ckpt["fusion"])
    fusion_net.eval()
    model = ImprovedToxicModel(
        total_feat_dim=GAN_INPUT_DIM,
        embed_dim=EMBED_DIM,
        exp_dim=EXP_DIM,
        species_num=species_num,
        num_classes=NUM_CLASSES
    ).to(device)
    model.load_state_dict(ckpt["model"])
    model.eval()
    return model, fusion_net, pca, scaler, exp_scaler, sp_enc

def run_single_prediction(df_input, assay_cols, model, fusion_net, pca, scaler, exp_scaler, sp_enc):
    train_sp_list = sp_enc.classes_
    feat_buffer = []
    sp_buffer = []
    exp_buffer = []

    for _, row in df_input.iterrows():
        f1a, f2a_raw, f3a, f4a, hb_a = extract_mol_features_raw(row["SMILES1"])
        f1b, f2b_raw, f3b, f4b, hb_b = extract_mol_features_raw(row["SMILES2"])
        f2a = pca.transform(f2a_raw.reshape(1, -1))[0]
        f2b = pca.transform(f2b_raw.reshape(1, -1))[0]

        assayA = row[assay_cols[:3]].values.astype(np.float32)
        assayB = row[assay_cols[3:]].values.astype(np.float32)
        inter_feat = calc_mol_mixed_interaction(f1a, f1b, f3a, f3b, f4a, f4b, hb_a, hb_b)

        modA = [
            torch.from_numpy(f1a).float().unsqueeze(0).to(device),
            torch.from_numpy(f2a).float().unsqueeze(0).to(device),
            torch.from_numpy(f3a).float().unsqueeze(0).to(device),
            torch.from_numpy(f4a).float().unsqueeze(0).to(device),
            torch.from_numpy(hb_a).float().unsqueeze(0).to(device),
            torch.from_numpy(assayA).float().unsqueeze(0).to(device)
        ]
        modB = [
            torch.from_numpy(f1b).float().unsqueeze(0).to(device),
            torch.from_numpy(f2b).float().unsqueeze(0).to(device),
            torch.from_numpy(f3b).float().unsqueeze(0).to(device),
            torch.from_numpy(f4b).float().unsqueeze(0).to(device),
            torch.from_numpy(hb_b).float().unsqueeze(0).to(device),
            torch.from_numpy(assayB).float().unsqueeze(0).to(device)
        ]
        with torch.no_grad():
            HA = fusion_net(modA).squeeze(0).cpu().numpy()
            HB = fusion_net(modB).squeeze(0).cpu().numpy()
        fusion_feat = np.concatenate([HA, HB, inter_feat]).reshape(1, -1)
        feat_buffer.append(fusion_feat)

        sp_raw = str(row["species"]).strip() if not pd.isna(row["species"]) else ""
        if sp_raw in train_sp_list:
            sp_id = sp_enc.transform([sp_raw])[0]
        else:
            rand_sp = np.random.choice(train_sp_list)
            sp_id = sp_enc.transform([rand_sp])[0]
        sp_buffer.append(sp_id)

        exp_val = np.array([row["exposure duration (days)"]], dtype=np.float32).reshape(-1,1)
        exp_buffer.append(exp_val)

    X_raw = np.vstack(feat_buffer)
    sp_arr = np.array(sp_buffer)
    exp_arr = np.vstack(exp_buffer)
    imputer = SimpleImputer(strategy="mean")
    X_clean = imputer.fit_transform(X_raw)
    X_norm = scaler.transform(X_clean)
    exp_clean = imputer.fit_transform(exp_arr)
    exp_norm = exp_scaler.transform(exp_clean)

    batch_size = 32
    n_sample = len(X_norm)
    all_prob = []
    all_pred_code = []
    with torch.no_grad():
        for start in range(0, n_sample, batch_size):
            end = min(start + batch_size, n_sample)
            ft_batch = torch.from_numpy(X_norm[start:end]).float().to(device)
            sp_batch = torch.from_numpy(sp_arr[start:end]).long().to(device)
            exp_batch = torch.from_numpy(exp_norm[start:end]).float().to(device)
            logits, _ = model(ft_batch, sp_batch, exp_batch)
            prob_mat = F.softmax(logits, dim=1).cpu().numpy()
            pred_cls = np.argmax(prob_mat, axis=1)
            all_prob.extend(prob_mat)
            all_pred_code.extend(pred_cls)
    all_prob = np.array(all_prob)
    pred_text = [inv_label_map[c] for c in all_pred_code]
    return all_pred_code, pred_text, all_prob

if __name__ == "__main__":
    PREDICT_FILE = "chemical-to-predict.xlsx"
    df_source = pd.read_excel(PREDICT_FILE, sheet_name=0)
    print(f"待预测数据集总样本数：{len(df_source)}")
    df_result = df_source.copy()

    for cfg in MODEL_CONFIGS:
        model_tag = cfg["name"]
        weight_path = cfg["ckpt_path"]
        assay_columns = cfg["assay_cols"]
        print(f"\n===== 开始加载 {model_tag} 模型：{weight_path} =====")
        model, fusion, pca, scaler, exp_scaler, spenc = load_model_ckpt(weight_path)
        pred_code, pred_text, prob_matrix = run_single_prediction(
            df_source, assay_columns, model, fusion, pca, scaler, exp_scaler, spenc
        )
        df_result[f"{model_tag}_pred_code"] = pred_code
        df_result[f"{model_tag}_pred_label"] = pred_text
        prob_df = pd.DataFrame(prob_matrix, columns=[f"{model_tag}_{cls}" for cls in class_names])
        df_result = pd.concat([df_result.reset_index(drop=True), prob_df.reset_index(drop=True)], axis=1)
        print(f"{model_tag} 模型预测完成")

    output_csv = "five_toxicity_joint_prediction.csv"
    df_result.to_csv(output_csv, index=False, encoding="utf-8-sig")
    print(f"\n全部5类毒性模型预测完成，结果保存至：{output_csv}")
    preview_cols = [
        "SMILES1", "SMILES2", "species", "exposure duration (days)",
        "endo_pred_label", "neuro_pred_label", "hep_pred_label", "imm_pred_label", "dev_pred_label"
    ]
    print("\n预测结果预览前10行：")
    print(df_result[preview_cols].head(10))